# Улучшение качества модели: Heart Disease (KNN + подбор гиперпараметров)

**Датасет:** `heart.csv`  
**Цель:** `HeartDisease` — 1 = есть заболевание сердца, 0 = нет.  
**Задача:** подготовить данные, обучить базовый KNN, подобрать гиперпараметры (GridSearchCV, RandomizedSearchCV, Hyperopt) и сравнить с другой моделью.


## 1. Получение и загрузка данных

In [1]:
# Импорты библиотек
import time  # для замера времени поиска гиперпараметров
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import (
    train_test_split,
    cross_val_score,
    StratifiedKFold,
    GridSearchCV,
    RandomizedSearchCV,
)
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report

# Hyperopt для байесовской оптимизации
from hyperopt import fmin, tpe, hp, Trials, STATUS_OK

RANDOM_STATE = 42  # фиксируем зерно, чтобы результаты повторялись
np.random.seed(RANDOM_STATE)


In [2]:
# Загружаем таблицу с признаками пациентов
df = pd.read_csv("heart.csv")

print("Размер датасета:", df.shape)
print("Колонки:", list(df.columns))
df.head()


Размер датасета: (918, 12)
Колонки: ['Age', 'Sex', 'ChestPainType', 'RestingBP', 'Cholesterol', 'FastingBS', 'RestingECG', 'MaxHR', 'ExerciseAngina', 'Oldpeak', 'ST_Slope', 'HeartDisease']


,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,40,M,ATA,140,289,0,Normal,172,N,0.0,Up,0
1,49,F,NAP,160,180,0,Normal,156,N,1.0,Flat,1
2,37,M,ATA,130,283,0,ST,98,N,0.0,Up,0
3,48,F,ASY,138,214,0,Normal,108,Y,1.5,Flat,1
4,54,M,NAP,150,195,0,Normal,122,N,0.0,Up,0


In [3]:
# Базовая информация: типы, пропуски, баланс классов
print("=== info ===")
df.info()
print("\n=== пропуски (NaN) ===")
print(df.isna().sum())
print("\n=== дубликаты ===")
print("Число полных дубликатов строк:", df.duplicated().sum())
print("\n=== целевая переменная HeartDisease ===")
print(df["HeartDisease"].value_counts())
print(df["HeartDisease"].value_counts(normalize=True).round(3))


=== info ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 918 entries, 0 to 917
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Age             918 non-null    int64  
 1   Sex             918 non-null    object 
 2   ChestPainType   918 non-null    object 
 3   RestingBP       918 non-null    int64  
 4   Cholesterol     918 non-null    int64  
 5   FastingBS       918 non-null    int64  
 6   RestingECG      918 non-null    object 
 7   MaxHR           918 non-null    int64  
 8   ExerciseAngina  918 non-null    object 
 9   Oldpeak         918 non-null    float64
 10  ST_Slope        918 non-null    object 
 11  HeartDisease    918 non-null    int64  
dtypes: float64(1), int64(6), object(5)
memory usage: 86.2+ KB

=== пропуски (NaN) ===
Age               0
Sex               0
ChestPainType     0
RestingBP         0
Cholesterol       0
FastingBS         0
RestingECG        0
MaxHR             0
Exerc

**Краткий вывод по шагу 1.**  
Датасет: **918** пациентов, **11** признаков + `HeartDisease`. Классы относительно сбалансированы (**55.3%** больных / **44.7%** здоровых). Явных NaN и дубликатов нет, но ниже проверим «скрытые» пропуски в виде нулей.


## 2. Подготовка датасета к обучению

### 2.1. Пропуски, дубликаты и «скрытые» нули

In [4]:
# В медицинских датасетах нули часто означают «не измерено», а не реальное значение
print("Cholesterol == 0:", (df["Cholesterol"] == 0).sum())
print("RestingBP == 0:", (df["RestingBP"] == 0).sum())

# Делаем копию, чтобы не портить исходную таблицу
data = df.copy()

# Нули в Cholesterol и RestingBP заменяем на NaN — так их видно как пропуски
data.loc[data["Cholesterol"] == 0, "Cholesterol"] = np.nan
data.loc[data["RestingBP"] == 0, "RestingBP"] = np.nan

print("\nПропуски после замены нулей на NaN:")
print(data.isna().sum())


Cholesterol == 0: 172
RestingBP == 0: 1

Пропуски после замены нулей на NaN:
Age                 0
Sex                 0
ChestPainType       0
RestingBP           1
Cholesterol       172
FastingBS           0
RestingECG          0
MaxHR               0
ExerciseAngina      0
Oldpeak             0
ST_Slope            0
HeartDisease        0
dtype: int64


In [5]:
# Заполняем числовые пропуски медианой (устойчивее среднего к выбросам)
for col in ["Cholesterol", "RestingBP"]:
    median_val = data[col].median()  # медиана по доступным значениям
    data[col] = data[col].fillna(median_val)  # подставляем медиану вместо NaN
    print(f"{col}: заполнено медианой = {median_val}")

# Удаляем полные дубликаты, если вдруг появятся
n_before = len(data)
data = data.drop_duplicates()
print(f"Удалено дубликатов: {n_before - len(data)}")
print("Пропусков осталось:", int(data.isna().sum().sum()))


Cholesterol: заполнено медианой = 237.0
RestingBP: заполнено медианой = 130.0
Удалено дубликатов: 0
Пропусков осталось: 0


### 2.2. Кодирование категориальных признаков

In [6]:
# Смотрим, какие значения принимают категориальные столбцы
cat_cols = ["Sex", "ChestPainType", "RestingECG", "ExerciseAngina", "ST_Slope"]
for c in cat_cols:
    print(f"{c}: {data[c].unique().tolist()}")


Sex: ['M', 'F']
ChestPainType: ['ATA', 'NAP', 'ASY', 'TA']
RestingECG: ['Normal', 'ST', 'LVH']
ExerciseAngina: ['N', 'Y']
ST_Slope: ['Up', 'Flat', 'Down']


In [7]:
# Отделяем целевую переменную
y = data["HeartDisease"]  # 0/1 — болезнь сердца
X = data.drop(columns=["HeartDisease"])  # все признаки без цели

# One-hot кодирование: каждая категория → отдельный столбец 0/1
# drop_first=True удалаяет первый столбец, чтобы избежать мультиколлинеарности
X_enc = pd.get_dummies(X, columns=cat_cols, drop_first=True)

print("Размер после кодирования:", X_enc.shape)
print("Признаки:", list(X_enc.columns))
X_enc.head()


Размер после кодирования: (918, 15)
Признаки: ['Age', 'RestingBP', 'Cholesterol', 'FastingBS', 'MaxHR', 'Oldpeak', 'Sex_M', 'ChestPainType_ATA', 'ChestPainType_NAP', 'ChestPainType_TA', 'RestingECG_Normal', 'RestingECG_ST', 'ExerciseAngina_Y', 'ST_Slope_Flat', 'ST_Slope_Up']


,Age,RestingBP,Cholesterol,FastingBS,MaxHR,Oldpeak,Sex_M,ChestPainType_ATA,ChestPainType_NAP,ChestPainType_TA,RestingECG_Normal,RestingECG_ST,ExerciseAngina_Y,ST_Slope_Flat,ST_Slope_Up
0,40,140.0,289.0,0,172,0.0,True,True,False,False,True,False,False,False,True
1,49,160.0,180.0,0,156,1.0,False,False,True,False,True,False,False,True,False
2,37,130.0,283.0,0,98,0.0,True,True,False,False,False,True,False,False,True
3,48,138.0,214.0,0,108,1.5,False,False,False,False,True,False,True,True,False
4,54,150.0,195.0,0,122,0.0,True,False,True,False,True,False,False,False,True


**Почему `get_dummies`, а не LabelEncoder?**  
LabelEncoder даёт числа 0, 1, 2… — для KNN это плохо: модель решит, что «2 ближе к 1, чем к 0», хотя категории несравнимы по порядку. One-hot честнее для расстояний.


### 2.3. Масштабирование признаков (важно для KNN)

In [8]:
# Числовые признаки имеют разный масштаб (Age ~50, Cholesterol ~200, Oldpeak ~1).
# KNN считает евклидово расстояние — без масштаба «победит» признак с большими числами.
num_cols = ["Age", "RestingBP", "Cholesterol", "FastingBS", "MaxHR", "Oldpeak"]

# Пока только готовим scaler; fit делаем на train после split (чтобы не было утечки данных)
scaler = StandardScaler()


**Как масштабирование влияет на KNN.**  
Без `StandardScaler` признаки вроде Cholesterol и RestingBP доминируют в расстоянии, а Oldpeak и FastingBS почти не участвуют. После стандартизации все признаки вносят вклад более равномерно — accuracy у KNN обычно заметно растёт. Конкретный прирост посчитаем в шаге 4.


## 3. Разделение выборки

In [9]:
# 80% — обучение, 20% — тест; stratify сохраняет долю больных/здоровых в обеих частях
X_train, X_test, y_train, y_test = train_test_split(
    X_enc,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

print("Train:", X_train.shape, "Test:", X_test.shape)
print("Доля больных в train:", round(y_train.mean(), 3))
print("Доля больных в test:", round(y_test.mean(), 3))


Train: (734, 15) Test: (184, 15)
Доля больных в train: 0.553
Доля больных в test: 0.554


In [10]:
# Масштабируем: учим параметры только на train, применяем к train и test
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)  # считаем mean/std по train
X_test_scaled = scaler.transform(X_test)        # теми же mean/std преобразуем test

# Для удобства кросс-валидации на всём train держим и немасштабированную копию
# (Pipeline внутри CV сам масштабирует каждый фолд — так правильнее)


**Краткий вывод по шагу 3.**  
Выборка: train **734**, test **184** (80/20). Доля больных почти одинакова (**0.553** / **0.554**) — стратификация сработала. Масштабирование учим только на train (в Pipeline внутри CV), чтобы не было утечки.


## 4. Базовая модель KNN и кросс‑валидация

Чтобы решить, болен ли новый пациент, смотрим на **k ближайших** уже известных пациентов (по признакам: возраст, давление, холестерин, ЭКГ и т.д.) и берём **большинство голосов** среди них.

- `n_neighbors = k` — сколько соседей спрашиваем. Малый k → слушаем «шум» (риск переобучения). Большой k → ответ слишком «средний» (риск недообучения).
- `weights='uniform'` — все соседи равны; `'distance'` — ближние голосуют сильнее.
- Расстояние считается в пространстве признаков → **масштаб важен**: без `StandardScaler` холестерин (~200) перекричит Oldpeak (~1).

**Кросс-валидация здесь** — несколько раз делим train на части и проверяем accuracy, чтобы понять: модель стабильна или качество «прыгает» от разбиения.


In [11]:
# Стратифицированная 5-fold CV: в каждом фолде сохраняем баланс классов
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# Pipeline: сначала StandardScaler, потом KNN — масштабирование внутри каждого фолда
pipe_knn_default = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier()),  # параметры по умолчанию: n_neighbors=5, weights='uniform'
])

# cross_val_score считает accuracy на каждом фолде
cv_scores_default = cross_val_score(
    pipe_knn_default,
    X_train,
    y_train,
    cv=cv,
    scoring="accuracy",
)

print("Accuracy по фолдам:", np.round(cv_scores_default, 4))
print("Среднее:", round(cv_scores_default.mean(), 4))
print("Стд. отклонение:", round(cv_scores_default.std(), 4))
print("Разброс (max - min):", round(cv_scores_default.max() - cv_scores_default.min(), 4))


Accuracy по фолдам: [0.8435 0.7959 0.8231 0.8639 0.8767]
Среднее: 0.8406
Стд. отклонение: 0.0288
Разброс (max - min): 0.0808


In [12]:
# Дополнительно: F1 и ROC-AUC той же CV
cv_f1 = cross_val_score(pipe_knn_default, X_train, y_train, cv=cv, scoring="f1")
cv_auc = cross_val_score(pipe_knn_default, X_train, y_train, cv=cv, scoring="roc_auc")
print("F1 mean ± std:", round(cv_f1.mean(), 4), "(стд отклонение)±", round(cv_f1.std(), 4))
print("ROC-AUC mean ± std:", round(cv_auc.mean(), 4), "(стд отклонение)±", round(cv_auc.std(), 4))


F1 mean ± std: 0.8572 (стд отклонение)± 0.0248
ROC-AUC mean ± std: 0.8977 (стд отклонение)± 0.0268


In [13]:
# Сравнение: KNN БЕЗ масштабирования vs С масштабированием
knn_no_scale = KNeighborsClassifier()  # сырые признаки
scores_no_scale = cross_val_score(knn_no_scale, X_train, y_train, cv=cv, scoring="accuracy")

print("KNN без масштабирования: mean =", round(scores_no_scale.mean(), 4),
      "std =", round(scores_no_scale.std(), 4))
print("KNN с масштабированием:  mean =", round(cv_scores_default.mean(), 4),
      "std =", round(cv_scores_default.std(), 4))
print("Прирост accuracy за счёт StandardScaler:",
      round(cv_scores_default.mean() - scores_no_scale.mean(), 4))


KNN без масштабирования: mean = 0.6553 std = 0.0461
KNN с масштабированием:  mean = 0.8406 std = 0.0288
Прирост accuracy за счёт StandardScaler: 0.1853


In [14]:
# Обучаем базовый KNN на всём train и смотрим качество на test
pipe_knn_default.fit(X_train, y_train)
y_pred_base = pipe_knn_default.predict(X_test)
y_proba_base = pipe_knn_default.predict_proba(X_test)[:, 1]

print("Test accuracy (базовый KNN):", round(accuracy_score(y_test, y_pred_base), 4))
print("Test F1:", round(f1_score(y_test, y_pred_base), 4))
print("Test ROC-AUC:", round(roc_auc_score(y_test, y_proba_base), 4))
print("\nTrain accuracy:", round(pipe_knn_default.score(X_train, y_train), 4))


Test accuracy (базовый KNN): 0.8913
Test F1: 0.9029
Test ROC-AUC: 0.9355

Train accuracy: 0.8842


**Вывод по шагу 4 (базовый KNN).**  
- CV accuracy: **0.841 ± 0.029**, разброс по фолдам ≈ **0.081** — модель **относительно устойчива**.  
- Train accuracy **0.884**, test **0.891** — test не хуже train, явного переобучения нет.  
- **Масштабирование критично:** без scaler CV ≈ **0.655**, с scaler ≈ **0.841** (прирост **~0.19**).  
- Дополнительно: F1 ≈ **0.86**, ROC-AUC ≈ **0.90** на CV.


## 5. Подбор гиперпараметров KNN: GridSearchCV и RandomizedSearchCV

### 5.1. GridSearchCV — полный перебор сетки

In [15]:
# Сетка гиперпараметров: что будем перебирать
param_grid = {
    "knn__n_neighbors": list(range(1, 31)),  # проверяем k от 1 до 30 (префикс knn__ — параметр шага Pipeline)
    "knn__weights": ["uniform", "distance"],  # uniform: все соседи равны; distance: ближние важнее
}

# Конвейер: сначала масштабируем признаки, потом обучаем KNN
pipe_knn = Pipeline([
    ("scaler", StandardScaler()),  # приводит признаки к одному масштабу
    ("knn", KNeighborsClassifier()),  # классификатор по ближайшим соседям
])

# Полный перебор всех комбинаций из сетки с кросс-валидацией
grid = GridSearchCV(
    estimator=pipe_knn,  # какую модель/пайплайн настраиваем
    param_grid=param_grid,  # какие значения гиперпараметров перебираем
    scoring="accuracy",  # чем измеряем качество на фолдах
    cv=cv,  # схема разбиения (у нас StratifiedKFold)
    n_jobs=-1,  # считать на всех ядрах процессора
    return_train_score=True,  # также сохранить качество на train (для оценки переобучения)
)

t0 = time.time()  # засекаем старт
grid.fit(X_train, y_train)  # перебираем все комбинации на CV и выбираем лучшую
grid_time = time.time() - t0  # сколько секунд занял поиск

print("Лучшие параметры (GridSearchCV):", grid.best_params_)
print("Лучшая CV accuracy:", round(grid.best_score_, 4))
print("Время работы, сек:", round(grid_time, 2))
print("Число комбинаций:", len(grid.cv_results_["params"]))


Лучшие параметры (GridSearchCV): {'knn__n_neighbors': 15, 'knn__weights': 'distance'}
Лучшая CV accuracy: 0.8665
Время работы, сек: 3.33
Число комбинаций: 60


In [16]:
# Качество лучшей модели Grid на отложенном тесте
best_grid = grid.best_estimator_
y_pred_grid = best_grid.predict(X_test)
print("Test accuracy (Grid KNN):", round(accuracy_score(y_test, y_pred_grid), 4))
print("Test F1:", round(f1_score(y_test, y_pred_grid), 4))
print("Test ROC-AUC:", round(roc_auc_score(y_test, best_grid.predict_proba(X_test)[:, 1]), 4))

# Смотрим train score лучшей конфигурации — намёк на переобучение
best_idx = grid.best_index_
print("Mean train score (лучшая):", round(grid.cv_results_["mean_train_score"][best_idx], 4))
print("Mean test score CV (лучшая):", round(grid.cv_results_["mean_test_score"][best_idx], 4))


Test accuracy (Grid KNN): 0.875
Test F1: 0.8867
Test ROC-AUC: 0.9378
Mean train score (лучшая): 1.0
Mean test score CV (лучшая): 0.8665


### 5.2. RandomizedSearchCV — случайный поиск

In [17]:
# Распределения параметров: случайно берём n_iter комбинаций
param_dist = {
    "knn__n_neighbors": list(range(1, 51)),  # более широкое пространство, чем у Grid
    "knn__weights": ["uniform", "distance"],
}

random_search = RandomizedSearchCV(
    estimator=pipe_knn,
    param_distributions=param_dist,
    n_iter=40,  # ограничение числа попыток
    scoring="accuracy",
    cv=cv,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    return_train_score=True,
)

t0 = time.time()
random_search.fit(X_train, y_train)
rand_time = time.time() - t0

print("Лучшие параметры (RandomizedSearchCV):", random_search.best_params_)
print("Лучшая CV accuracy:", round(random_search.best_score_, 4))
print("Время работы, сек:", round(rand_time, 2))
print("Число итераций:", random_search.n_iter)


Лучшие параметры (RandomizedSearchCV): {'knn__weights': 'uniform', 'knn__n_neighbors': 15}
Лучшая CV accuracy: 0.8638
Время работы, сек: 1.14
Число итераций: 40


In [18]:
# Тест для лучшей модели RandomizedSearchCV
best_rand = random_search.best_estimator_
y_pred_rand = best_rand.predict(X_test)
print("Test accuracy (Random KNN):", round(accuracy_score(y_test, y_pred_rand), 4))
print("Test F1:", round(f1_score(y_test, y_pred_rand), 4))
print("Test ROC-AUC:", round(roc_auc_score(y_test, best_rand.predict_proba(X_test)[:, 1]), 4))


Test accuracy (Random KNN): 0.8696
Test F1: 0.8824
Test ROC-AUC: 0.9331


### 5.3. Сравнение GridSearchCV и RandomizedSearchCV

In [19]:
# Сводная таблица двух методов поиска
compare_search = pd.DataFrame({
    "метод": ["GridSearchCV", "RandomizedSearchCV"],
    "лучшие_параметры": [str(grid.best_params_), str(random_search.best_params_)],
    "CV_accuracy": [grid.best_score_, random_search.best_score_],
    "test_accuracy": [
        accuracy_score(y_test, y_pred_grid),
        accuracy_score(y_test, y_pred_rand),
    ],
    "время_сек": [grid_time, rand_time],
    "попыток": [len(grid.cv_results_["params"]), random_search.n_iter],
})
print(compare_search.round(4).to_string(index=False))


             метод                                     лучшие_параметры  CV_accuracy  test_accuracy  время_сек  попыток
      GridSearchCV {'knn__n_neighbors': 15, 'knn__weights': 'distance'}       0.8665         0.8750     3.3288       60
RandomizedSearchCV  {'knn__weights': 'uniform', 'knn__n_neighbors': 15}       0.8638         0.8696     1.1350       40


**Вывод по шагу 5.**  
- **GridSearchCV** перебрал все варианты и выбрал: 15 соседей, веса `distance`. Качество на CV ≈ **0.867**, на тесте ≈ **0.875**, заняло около **3 секунд** (60 комбинаций).  
- **RandomizedSearchCV** пробовал наугад и нашёл: тоже 15 соседей, но веса `uniform`. CV ≈ **0.864**, тест ≈ **0.870**, заняло около **1 секунды** (40 попыток).  
- Оба метода сошлись на одном числе соседей (**k=15**). Тип весов разный, но качество почти одинаковое — разница крошечная.  
- Случайный поиск оказался примерно в **3 раза быстрее**, а результат почти не хуже.  
- У варианта с `distance` на обучающей выборке accuracy = **1.0**: факт переобучения.
- Если соседей слишком мало — модель цепляется за шум (переобучение). Если слишком много — ответ становится слишком «средним» (недообучение).


## 6. Сравнение с логистической регрессией

In [20]:
pipe_logreg = Pipeline([
    ("scaler", StandardScaler()),
    ("logreg", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
])

cv_logreg = cross_val_score(pipe_logreg, X_train, y_train, cv=cv, scoring="accuracy")
cv_logreg_f1 = cross_val_score(pipe_logreg, X_train, y_train, cv=cv, scoring="f1")
cv_logreg_auc = cross_val_score(pipe_logreg, X_train, y_train, cv=cv, scoring="roc_auc")

print("LogReg CV accuracy по фолдам:", np.round(cv_logreg, 4))
print("LogReg mean ± std:", round(cv_logreg.mean(), 4), "±", round(cv_logreg.std(), 4))
print("LogReg разброс (max-min):", round(cv_logreg.max() - cv_logreg.min(), 4))
print("LogReg F1 mean:", round(cv_logreg_f1.mean(), 4))
print("LogReg ROC-AUC mean:", round(cv_logreg_auc.mean(), 4))


LogReg CV accuracy по фолдам: [0.8367 0.7959 0.8571 0.8844 0.863 ]
LogReg mean ± std: 0.8474 ± 0.0299
LogReg разброс (max-min): 0.0884
LogReg F1 mean: 0.8633
LogReg ROC-AUC mean: 0.922


In [21]:
# Обучение LogReg и оценка на тесте
pipe_logreg.fit(X_train, y_train)
y_pred_lr = pipe_logreg.predict(X_test)
print("Test accuracy (LogReg):", round(accuracy_score(y_test, y_pred_lr), 4))
print("Test F1:", round(f1_score(y_test, y_pred_lr), 4))
print("Test ROC-AUC:", round(roc_auc_score(y_test, pipe_logreg.predict_proba(X_test)[:, 1]), 4))
print("\nclassification_report:\n")
print(classification_report(y_test, y_pred_lr, digits=3))


Test accuracy (LogReg): 0.8913
Test F1: 0.9029
Test ROC-AUC: 0.9327

classification_report:

              precision    recall  f1-score   support

           0      0.887     0.866     0.877        82
           1      0.894     0.912     0.903       102

    accuracy                          0.891       184
   macro avg      0.891     0.889     0.890       184
weighted avg      0.891     0.891     0.891       184



In [22]:
# Сводка: базовый KNN vs улучшенный KNN (Grid) vs LogReg
summary_models = pd.DataFrame({
    "модель": [
        "KNN default",
        "KNN GridSearch",
        "LogisticRegression",
    ],
    "CV_accuracy_mean": [
        cv_scores_default.mean(),
        grid.best_score_,
        cv_logreg.mean(),
    ],
    "CV_accuracy_std": [
        cv_scores_default.std(),
        grid.cv_results_["std_test_score"][grid.best_index_],
        cv_logreg.std(),
    ],
    "test_accuracy": [
        accuracy_score(y_test, y_pred_base),
        accuracy_score(y_test, y_pred_grid),
        accuracy_score(y_test, y_pred_lr),
    ],
})
print(summary_models.round(4).to_string(index=False))


            модель  CV_accuracy_mean  CV_accuracy_std  test_accuracy
       KNN default            0.8406           0.0288         0.8913
    KNN GridSearch            0.8665           0.0305         0.8750
LogisticRegression            0.8474           0.0299         0.8913


**Вывод по шагу 6.**  
- Логистическая регрессия на тесте дала accuracy **0.891** — как у базового KNN, и не хуже улучшенного KNN.  
- На кросс-валидации Grid-KNN чуть выше (**0.867** против **0.847**), но разница небольшая.  
- **Для практики лучше LogReg:** качество почти то же, а модель проще понять и быстрее работает. KNN оставляем, если важны именно «похожие пациенты».


## 7. Байесовская оптимизация гиперпараметров (Hyperopt / TPE)

Обычный Grid перебирает **все** варианты. Random пробует **наугад**.  

**Hyperopt (TPE)** делает умнее: после каждой попытки смотрит, какие настройки уже давали хороший результат, и в следующий раз чаще проверяет похожие.  

На нашей задаче это поиск лучших `n_neighbors` и `weights` для KNN: хотим высокую accuracy на кросс-валидации, тратя меньше лишних запусков.


In [23]:
# Пространство поиска для Hyperopt
# hp.quniform даёт «почти целые» значения; потом приводим к int
space = {
    "n_neighbors": hp.quniform("n_neighbors", 1, 50, 1),
    "weights": hp.choice("weights", ["uniform", "distance"]),
}


In [24]:
# Целевая функция: Hyperopt МИНИМИЗИРУЕТ значение,
# поэтому возвращаем отрицательную accuracy (чем выше accuracy — тем меньше loss)
def objective(params):
    n_neighbors = int(params["n_neighbors"])  # quniform возвращает float
    weights = params["weights"]

    model = Pipeline([
        ("scaler", StandardScaler()),
        ("knn", KNeighborsClassifier(n_neighbors=n_neighbors, weights=weights)),
    ])

    scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring="accuracy",
    )
    # STATUS_OK — стандартный статус успешной оценки для Trials
    return {
        "loss": -scores.mean(),  # отрицательное среднее accuracy
        "status": STATUS_OK,
        "accuracy_mean": scores.mean(),
        "accuracy_std": scores.std(),
    }


In [25]:
# Trials хранит историю всех попыток
trials = Trials()
MAX_EVALS = 40  # сколько раз вызываем objective

t0 = time.time()
best_hyperopt = fmin(
    fn=objective,          # целевая функция
    space=space,           # пространство гиперпараметров
    algo=tpe.suggest,      # Tree-structured Parzen Estimator (байесовский подход)
    max_evals=MAX_EVALS,
    trials=trials,
    rstate=np.random.default_rng(RANDOM_STATE),
)
hyperopt_time = time.time() - t0

# hp.choice возвращает индекс — переводим обратно в строку
weights_options = ["uniform", "distance"]
best_n = int(best_hyperopt["n_neighbors"])
best_w = weights_options[best_hyperopt["weights"]]

print("Лучшие параметры (Hyperopt):", {"n_neighbors": best_n, "weights": best_w})
print("Лучшая CV accuracy:", round(-trials.best_trial["result"]["loss"], 4))
print("Итераций (max_evals):", MAX_EVALS)
print("Время работы, сек:", round(hyperopt_time, 2))


100%|██████████| 40/40 [00:01<00:00, 25.57trial/s, best loss: -0.862426614481409]
Лучшие параметры (Hyperopt): {'n_neighbors': 42, 'weights': 'uniform'}
Лучшая CV accuracy: 0.8624
Итераций (max_evals): 40
Время работы, сек: 1.57


In [26]:
# Обучаем KNN с параметрами Hyperopt и проверяем на тесте
pipe_hyperopt = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(n_neighbors=best_n, weights=best_w)),
])
pipe_hyperopt.fit(X_train, y_train)
y_pred_hp = pipe_hyperopt.predict(X_test)

print("Test accuracy (Hyperopt KNN):", round(accuracy_score(y_test, y_pred_hp), 4))
print("Test F1:", round(f1_score(y_test, y_pred_hp), 4))
print("Test ROC-AUC:", round(roc_auc_score(y_test, pipe_hyperopt.predict_proba(X_test)[:, 1]), 4))


Test accuracy (Hyperopt KNN): 0.8913
Test F1: 0.902
Test ROC-AUC: 0.9415


In [27]:
# Сравнение трёх методов улучшенной KNN
tune_compare = pd.DataFrame({
    "метод": ["GridSearchCV", "RandomizedSearchCV", "Hyperopt (TPE)"],
    "n_neighbors": [
        grid.best_params_["knn__n_neighbors"],
        random_search.best_params_["knn__n_neighbors"],
        best_n,
    ],
    "weights": [
        grid.best_params_["knn__weights"],
        random_search.best_params_["knn__weights"],
        best_w,
    ],
    "CV_accuracy": [
        grid.best_score_,
        random_search.best_score_,
        -trials.best_trial["result"]["loss"],
    ],
    "test_accuracy": [
        accuracy_score(y_test, y_pred_grid),
        accuracy_score(y_test, y_pred_rand),
        accuracy_score(y_test, y_pred_hp),
    ],
    "время_сек": [grid_time, rand_time, hyperopt_time],
    "попыток": [
        len(grid.cv_results_["params"]),
        random_search.n_iter,
        MAX_EVALS,
    ],
})
print(tune_compare.round(4).to_string(index=False))


             метод  n_neighbors  weights  CV_accuracy  test_accuracy  время_сек  попыток
      GridSearchCV           15 distance       0.8665         0.8750     3.3288       60
RandomizedSearchCV           15  uniform       0.8638         0.8696     1.1350       40
    Hyperopt (TPE)           42  uniform       0.8624         0.8913     1.5709       40


**Вывод по шагу 7.**  
- Hyperopt нашёл: **42** соседа, веса `uniform`. CV ≈ **0.862**, тест ≈ **0.891**, за **40** попыток (~1.5 с).  
- Качество почти как у Grid и Random. Параметры не совпали один в один (у Grid было k=15), но результат очень близкий.  
- Плюс такого поиска: он не тыкает совсем наугад и не перебирает всё подряд, а чаще пробует то, что уже хорошо себя показало.


## 8. Итоговые выводы

In [28]:
# Финальная сводная таблица всех моделей
final_table = pd.DataFrame({
    "модель": [
        "KNN default (k=5)",
        "KNN GridSearchCV",
        "KNN RandomizedSearchCV",
        "KNN Hyperopt (TPE)",
        "LogisticRegression",
    ],
    "CV_accuracy": [
        cv_scores_default.mean(),
        grid.best_score_,
        random_search.best_score_,
        -trials.best_trial["result"]["loss"],
        cv_logreg.mean(),
    ],
    "CV_std": [
        cv_scores_default.std(),
        grid.cv_results_["std_test_score"][grid.best_index_],
        random_search.cv_results_["std_test_score"][random_search.best_index_],
        trials.best_trial["result"]["accuracy_std"],
        cv_logreg.std(),
    ],
    "test_accuracy": [
        accuracy_score(y_test, y_pred_base),
        accuracy_score(y_test, y_pred_grid),
        accuracy_score(y_test, y_pred_rand),
        accuracy_score(y_test, y_pred_hp),
        accuracy_score(y_test, y_pred_lr),
    ],
})
print(final_table.round(4).to_string(index=False))


                модель  CV_accuracy  CV_std  test_accuracy
     KNN default (k=5)       0.8406  0.0288         0.8913
      KNN GridSearchCV       0.8665  0.0305         0.8750
KNN RandomizedSearchCV       0.8638  0.0333         0.8696
    KNN Hyperopt (TPE)       0.8624  0.0289         0.8913
    LogisticRegression       0.8474  0.0299         0.8913


### Итоговый вывод

1. **Данные.** Скрытые пропуски (нули) заполнили медианой, категории закодировали, признаки масштабировали. Без масштабирования KNN сильно хуже (~0.66 → ~0.84).

2. **Базовый KNN** показывает хороший резульат (test ≈ **0.89**) и довольно устойчив.

3. **Подбор настроек** (Grid / Random / Hyperopt) чуть поднял CV, но на тесте выигрыш небольшой. Все три способа дали похожее качество; Random и Hyperopt быстрее полного перебора.

4. **Мало соседей** — риск переобучения, **слишком много** — недообучение. Ориентир у нас примерно **15–40** соседей.

5. **На практику:** берём **логистическую регрессию** — test ≈ **0.89**, проще и понятнее. Если нужен KNN — только со `StandardScaler` и подобранным k.
